# 00 — Data Acquisition
Download the **Credit Card Transactions Fraud Detection Dataset** from Kaggle
(`kartik2112/fraud-detection`) and do a basic sanity check.

Dataset URL: https://www.kaggle.com/datasets/kartik2112/fraud-detection

In [1]:
import os, subprocess, pathlib

RAW_DIR = pathlib.Path('data/raw')
RAW_DIR.mkdir(parents=True, exist_ok=True)

train_path = RAW_DIR / 'fraudTrain.csv'
test_path  = RAW_DIR / 'fraudTest.csv'

if train_path.exists() and test_path.exists():
    print('Files already present — skipping download.')
else:
    print('Downloading from Kaggle...')
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'kartik2112/fraud-detection',
         '-p', str(RAW_DIR), '--unzip'],
        check=True
    )
    print('Done.')

print(f'Train: {train_path}  ({train_path.stat().st_size / 1e6:.1f} MB)')
print(f'Test : {test_path}   ({test_path.stat().st_size  / 1e6:.1f} MB)')

Files already present — skipping download.
Train: data/raw/fraudTrain.csv  (351.2 MB)
Test : data/raw/fraudTest.csv   (150.4 MB)


In [2]:
import pandas as pd

train = pd.read_csv(train_path, index_col=0)
test  = pd.read_csv(test_path,  index_col=0)

print('=== fraudTrain.csv ===')
print(f'Shape : {train.shape}')
print(f'Columns: {list(train.columns)}')
train.head(3)

=== fraudTrain.csv ===
Shape : (1296675, 22)
Columns: ['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud']


,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,...,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,...,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,...,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0


In [3]:
print('=== Basic stats ===')
print(f'Unique customers (cc_num) : {train["cc_num"].nunique():,}')
print(f'Unique merchants          : {train["merchant"].nunique():,}')
print(f'Unique categories         : {train["category"].nunique()}')
print(f'Date range                : {train["trans_date_trans_time"].min()} → {train["trans_date_trans_time"].max()}')
print(f'Fraud rate                : {train["is_fraud"].mean()*100:.2f}%')
print(f'Missing values:\n{train.isnull().sum()[train.isnull().sum() > 0]}')

=== Basic stats ===
Unique customers (cc_num) : 983
Unique merchants          : 693
Unique categories         : 14
Date range                : 2019-01-01 00:00:18 → 2020-06-21 12:13:37
Fraud rate                : 0.58%
Missing values:
Series([], dtype: int64)


In [4]:
# Transactions per customer distribution
txn_per_customer = train.groupby('cc_num').size()
print(f'Transactions per customer — min: {txn_per_customer.min()}, '
      f'median: {txn_per_customer.median():.0f}, '
      f'max: {txn_per_customer.max()}')
print(f'Customers with < 10 txns  : {(txn_per_customer < 10).sum()}')

Transactions per customer — min: 7, median: 1054, max: 3123
Customers with < 10 txns  : 35


In [5]:
print('All 14 categories:')
print(sorted(train['category'].unique()))

All 14 categories:
['entertainment', 'food_dining', 'gas_transport', 'grocery_net', 'grocery_pos', 'health_fitness', 'home', 'kids_pets', 'misc_net', 'misc_pos', 'personal_care', 'shopping_net', 'shopping_pos', 'travel']
